# 원본 기반 파생변수와 동일 모델 비교
먼저 10_original_defect_report_revision.ipynb로 원본 Shot별 과거 센서 피처를 생성합니다. 같은 Random Forest로 기본 센서, 센서 조합, 최근 변화/변동폭, 전체 파생변수를 비교합니다. 전체 불량과 가스 타깃을 별도로 평가합니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut

root=Path.cwd()
if not (root/'data').exists():
    root=root.parent
tables=root/'reports/tables'
data=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
raw=pd.read_csv(root/'data/moldset_labeled_cn7.csv',nrows=0).columns.drop(['Unnamed: 0','PassOrFail']).tolist()
assert len(data)==1304 and data.PassOrFail.sum()==44
assert data[['product','TimeStamp']].duplicated().sum()==0

X=data[raw].copy()
X['product_RG3']=data['product'].eq('RG3').astype(int)
base=list(X.columns)
def safe_div(a,b):
    return a.div(b.where(b.abs().gt(1e-6))).replace([np.inf,-np.inf],np.nan)
X['fill_fraction_of_injection']=safe_div(X.Filling_Time,X.Injection_Time)
X['injection_minus_filling']=X.Injection_Time-X.Filling_Time
X['plasticizing_fraction_of_cycle']=safe_div(X.Plasticizing_Time,X.Cycle_Time)
X['filling_fraction_of_cycle']=safe_div(X.Filling_Time,X.Cycle_Time)
X['speed_times_filling']=X.Max_Injection_Speed*X.Filling_Time
X['injection_pressure_gap']=X.Max_Injection_Pressure-X.Max_Switch_Over_Pressure
X['mold_temperature_gap']=X.Mold_Temperature_4-X.Mold_Temperature_3
barrels=['Barrel_Temperature_'+str(k) for k in range(1,7)]
X['barrel_temperature_mean']=X[barrels].mean(axis=1)
X['barrel_temperature_spread']=X[barrels].std(axis=1)
X['barrel_temperature_1_minus_5']=X.Barrel_Temperature_1-X.Barrel_Temperature_5
process=[f for f in X if f not in base]
history=[]
for feature in raw:
    for suffix in ['__delta1','__dev10','__past10_std']:
        name=feature+suffix
        X[name]=data[name]
        history.append(name)
    name=feature+'__relative_deviation10'
    X[name]=safe_div(data[feature+'__dev10'],data[feature+'__past10_std']).clip(-20,20)
    history.append(name)
X=X.replace([np.inf,-np.inf],np.nan)
sets={
    'baseline':base,
    'process_combinations':base+process,
    'recent_history':base+history,
    'all_derived':base+process+history,
}
targets={'all_defects':data.PassOrFail.astype(int),
         'gas':data.Reason.str.contains('가스',regex=False).astype(int)}
assert targets['gas'].sum()==30
groups=data['comparison_block']
tie_breaker=np.random.RandomState(42).random_sample(len(data))
folds=list(LeaveOneGroupOut().split(X,targets['all_defects'],groups))
assert len(folds)==8
for train,test in folds:
    assert set(groups.iloc[train]).isdisjoint(groups.iloc[test])
# No Reason, side-specific defect labels, dates, block IDs, or normal-block baselines enter X.
for forbidden in ['Reason','PassOrFail','TimeStamp','comparison_block','bad_side_reason']:
    assert forbidden not in X
X.assign(product=data['product'],TimeStamp=data['TimeStamp'],comparison_block=groups,
         PassOrFail=targets['all_defects'],gas=targets['gas']).to_csv(
    tables/'engineered_shot_features.csv',index=False,encoding='utf-8-sig')
print('Feature counts:', {name:len(cols) for name,cols in sets.items()})
print('Targets:', {name:int(y.sum()) for name,y in targets.items()}, 'shots=',len(data))

In [ ]:
metrics, predictions, importances, block_metrics, types = [], [], [], [], []
for target,y in targets.items():
    for setting,cols in sets.items():
        probability=np.full(len(data),np.nan)
        importance=np.zeros(len(cols))
        block_top10=np.zeros(len(data),dtype=bool)
        for fold,(train,test) in enumerate(folds):
            model=make_pipeline(
                SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True),
                RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,
                                       class_weight='balanced_subsample',random_state=42,n_jobs=-1))
            model.fit(X.iloc[train][cols],y.iloc[train])
            probability[test]=model.predict_proba(X.iloc[test][cols])[:,1]
            importance += model[-1].feature_importances_[:len(cols)] / len(folds)
            order=test[np.lexsort((tie_breaker[test],-probability[test]))]
            block_top10[order[:max(1,int(np.ceil(len(test)*.10)))]]=True
            predicted=probability[test]>=.5
            block_metrics.append({'target':target,'setting':setting,'comparison_block':groups.iloc[test[0]],
                                  'test_shots':len(test),'positive_shots':int(y.iloc[test].sum()),
                                  'caught_at_0_5':int((predicted & y.iloc[test].to_numpy().astype(bool)).sum()),
                                  'false_positive_at_0_5':int((predicted & ~y.iloc[test].to_numpy().astype(bool)).sum()),
                                  'caught_top10pct_in_block':int((block_top10[test] & y.iloc[test].to_numpy().astype(bool)).sum())})
        assert np.isfinite(probability).all()
        predicted=probability>=.5
        order=np.lexsort((tie_breaker,-probability))
        top10=np.zeros(len(data),dtype=bool)
        top10[order[:int(np.ceil(len(data)*.10))]]=True
        metric={'target':target,'setting':setting,'features':len(cols),'shots':len(data),'positives':int(y.sum()),
                'average_precision':average_precision_score(y,probability),
                'roc_auc':roc_auc_score(y,probability),
                'precision_at_0_5':precision_score(y,predicted,zero_division=0),
                'recall_at_0_5':recall_score(y,predicted,zero_division=0),
                'f1_at_0_5':f1_score(y,predicted,zero_division=0),
                'caught_at_0_5':int((predicted & y.astype(bool)).sum()),
                'false_positive_at_0_5':int((predicted & ~y.astype(bool)).sum()),
                'caught_global_top10pct':int((top10 & y.astype(bool)).sum()),
                'caught_block_top10pct':int((block_top10 & y.astype(bool)).sum()),
                'block_top10pct_selected':int(block_top10.sum())}
        metrics.append(metric)
        print(target,setting,'AP',round(metric['average_precision'],4),'caught',metric['caught_at_0_5'],
              'FP',metric['false_positive_at_0_5'],'top10/block',metric['caught_block_top10pct'],flush=True)
        for i in range(len(data)):
            predictions.append({'target':target,'setting':setting,'product':data.iloc[i]['product'],
                                'TimeStamp':data.iloc[i]['TimeStamp'],'comparison_block':groups.iloc[i],
                                'Reason':data.iloc[i]['Reason'],'y':int(y.iloc[i]),
                                'probability':probability[i],'prediction_0_5':int(predicted[i]),
                                'selected_top10pct_in_block':int(block_top10[i])})
        for feature,value in zip(cols,importance):
            importances.append({'target':target,'setting':setting,'feature':feature,'mean_tree_importance':value})
        for (product,reason),g in data[data.PassOrFail.eq(1)].groupby(['product','Reason']):
            idx=g.index.to_numpy()
            if target=='gas':
                idx=idx[y.iloc[idx].to_numpy().astype(bool)]
            if len(idx):
                types.append({'target':target,'setting':setting,'product':product,'Reason':reason,
                              'defect_shots':len(idx),'caught_at_0_5':int(predicted[idx].sum()),
                              'caught_top10pct_in_block':int(block_top10[idx].sum())})
metrics=pd.DataFrame(metrics)
pd.DataFrame(predictions).to_csv(tables/'feature_ablation_oof_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'feature_ablation_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(block_metrics).to_csv(tables/'feature_ablation_by_observation_block.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(types).to_csv(tables/'feature_ablation_by_defect_type.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(importances).to_csv(tables/'feature_ablation_importances.csv',index=False,encoding='utf-8-sig')
metrics

In [ ]:
# Optimistic reference: the same observation episode may appear in both train and test.
from sklearn.model_selection import StratifiedKFold
reference_metrics=[]
for target,y in targets.items():
    for setting in ['baseline','all_derived']:
        cols=sets[setting]
        probability=np.full(len(data),np.nan)
        for train,test in StratifiedKFold(n_splits=5,shuffle=True,random_state=42).split(X,y):
            model=make_pipeline(
                SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True),
                RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,
                                       class_weight='balanced_subsample',random_state=42,n_jobs=-1))
            model.fit(X.iloc[train][cols],y.iloc[train])
            probability[test]=model.predict_proba(X.iloc[test][cols])[:,1]
        selection=np.zeros(len(data),dtype=bool)
        for block in groups.unique():
            idx=np.flatnonzero(groups.eq(block).to_numpy())
            order=idx[np.lexsort((tie_breaker[idx],-probability[idx]))]
            selection[order[:max(1,int(np.ceil(len(idx)*.10)))]]=True
        predicted=probability>=.5
        reference_metrics.append({'target':target,'setting':setting,
            'average_precision':average_precision_score(y,probability),
            'caught_at_0_5':int((predicted & y.astype(bool)).sum()),
            'false_positive_at_0_5':int((predicted & ~y.astype(bool)).sum()),
            'caught_block_top10pct':int((selection & y.astype(bool)).sum()),'positives':int(y.sum())})
reference_metrics=pd.DataFrame(reference_metrics)
reference_metrics.to_csv(tables/'feature_ablation_random_split_reference.csv',index=False,encoding='utf-8-sig')
print('Optimistic within-episode reference:',reference_metrics.to_dict('records'))
names={'baseline':'기본 센서','process_combinations':'기본 + 센서 조합',
       'recent_history':'기본 + 최근 변화·변동폭','all_derived':'기본 + 모든 파생변수'}
lines=['# 파생변수 1차 비교 결과','',
       '주요 평가에서 파생변수의 새 관측 구간 탐지 개선은 확인되지 않았다. 구간별 상위10%를 검사하면 기본 센서/모든 파생변수의 전체 불량 발견은 8/44와 7/44, 가스 발견은 둘 다 2/30이었다. 같은 관측 사건을 공유하는 임의 분할 참고 시험에서는 전체 불량 12→18/44, 가스 5→9/30으로 늘었지만 새 구간 성능으로 해석할 수 없다. 고정 0.5 임계값에서는 모든 파생변수 모델의 발견이 0이므로 이 모델을 그대로 판별기로 사용할 수준은 아니다.','',
       '원본 제품별 라벨 1,304 Shot을 사용했다. 전체 불량 44 Shot, 가스는 혼합 불량을 포함한 30 Shot이다. RH/LH는 한 Shot으로 묶고, 같은 Shot에서 하나라도 불량이면 전체 불량 타깃을 1로 정의했다. 가스 타깃은 가스 여부이며 다른 불량은 가스 타깃에서 0이다.',
       '',
       '모델은 모든 실험에서 같은 Random Forest(트리 200개, 최대 깊이 8, 최소 말단 3 Shot, 클래스 가중치, 고정 시드)를 사용했다. CN7 3개·RG3 5개의 라벨 관측 구간 중 하나를 통째로 제외하고 나머지 7개로 학습하는 평가를 8번 실시했다. 결측 대체값은 학습 구간에서만 계산했다. 과거 피처에는 현재 Shot 이전의 센서값만 사용했고 불량 라벨은 넣지 않았다.',
       '',
       '관측 구간은 실제 Run 로그가 아니다. 학습 데이터에 평가 구간보다 나중 시점이 포함될 수 있어, 이 평가는 미래 생산을 순서대로 예측하는 성능과 같지 않다. 모든 파생변수 조합은 같은 평가 결과에서 비교했으므로 독립 최종 테스트는 아직 없다.',
       '',
       '## 만든 파생변수','',
       '- 최근 변화: 직전 Shot 대비 차이, 직전 최대 10 Shot 중앙값 대비 차이, 직전 최대 10 Shot 표준편차, 표준편차로 나눈 변화량. 유효 이력 5개 미만은 결측으로 남겼다.',
       '- 센서 조합: 충전/사출시간 비율, 사출-충전시간, 계량/사이클시간 비율, 충전/사이클시간 비율, 속도×충전시간, 사출-전환압력 차이, 금형온도4-3, 배럴온도 평균·온도 분포폭·1-5 차이.',
       '- 속도×충전시간은 탐색용 숫자 조합이며 유량이나 수지량을 직접 측정한 값으로 해석하지 않는다.',
       '- 날짜, 관측 구간 ID, 불량 사유, 정상/불량 비교용 중앙값은 모델 입력에서 제외했다.',
       '',
       '## 같은 모델의 성능 비교','',
       'AP는 불량을 위험도 순위 앞쪽에 배치하는 정도이며 높을수록 좋다. 0.5 기준은 조정하지 않은 고정 임계값이다. 상위10% 검사는 각 관측 구간 안에서 위험도가 높은 약 10%의 Shot을 선택한 결과다.',
       '',
       '| 타깃 | 입력 | AP | 0.5 기준 발견 | 정상/비가스 오탐 | 구간별 상위10%에서 발견 |',
       '|---|---|---:|---:|---:|---:|']
for r in metrics.itertuples(index=False):
    label='전체 불량' if r.target=='all_defects' else '가스'
    lines.append('| {} | {} | {:.3f} | {}/{} | {} | {}/{} |'.format(
        label,names[r.setting],r.average_precision,r.caught_at_0_5,r.positives,
        r.false_positive_at_0_5,r.caught_block_top10pct,r.positives))
lines+=['','## 유형별 발견 결과','','상위10% 검사 결과를 모든 파생변수 입력 기준으로 표시했다.','',
        '| 타깃 | 제품·불량 | Shot | 0.5 기준 발견 | 구간별 상위10% 발견 |',
        '|---|---|---:|---:|---:|']
for r in pd.DataFrame(types).query("setting == 'all_derived'").itertuples(index=False):
    lines.append('| {} | {} {} | {} | {} | {} |'.format(
        '전체 불량' if r.target=='all_defects' else '가스',r.product,r.Reason,
        r.defect_shots,r.caught_at_0_5,r.caught_top10pct_in_block))
lines+=['','## 같은 관측 사건을 공유하는 임의 분할: 낙관적인 참고 결과','', 'Shot 단위 임의 5분할이라 같은 연속 사건이 학습·평가에 함께 들어갈 수 있다. 새 관측 구간 성능으로 해석하지 않는다.', '', '| 타깃 | 입력 | AP | 0.5 기준 발견 | 구간별 상위10% 발견 |','|---|---|---:|---:|---:|']
for r in reference_metrics.itertuples(index=False):
    lines.append('| {} | {} | {:.3f} | {}/{} | {}/{} |'.format('전체 불량' if r.target=='all_defects' else '가스',names[r.setting],r.average_precision,r.caught_at_0_5,r.positives,r.caught_block_top10pct,r.positives))
lines+=['','## 해석','',
    '- 표본이 작고 CN7 가스·미성형은 한 관측 구간에 몰려 있다. 해당 구간을 통째로 평가하면 학습 데이터에 같은 CN7 사건이 없어 성능이 낮아질 수 있다. 이 실패도 평가 결과에 포함했다.',
    '- 임의 분할 결과는 같은 사건의 패턴을 학습한 경우의 참고값이다. 주요 성능 판단은 관측 구간 전체를 제외하는 평가로 한다.',
    '- 원본에는 대회 범위 밖의 초기허용불량이 포함돼 있다. 이 결과는 원본 데이터의 탐색 실험이며 대회 공식 성능은 아니다.',
    '- Timestamp와 직전 센서 이력이 실제 테스트에서 제공되지 않으면 최근 변화·변동폭 피처는 그대로 사용할 수 없다.',
    '- 다음 판단은 전체 평균만이 아니라 구간별·불량 유형별 실패 사례를 함께 기준으로 삼는다.']
(root/'reports/derived_feature_model_comparison.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('Saved model comparison report and all out-of-fold results.')